In [1]:
# ============================================================
# Section 1: Imports and Test 2 settings
# ============================================================

from pathlib import Path
from datetime import datetime

import osxphotos

from explorephotoslibrary import *

USE_INVENTORY_CACHE = True

# Rebuild only the libraries listed here.
# Usually:
# FORCE_REBUILD_INVENTORY_KEYS = set()
#
# If you changed both backup and current Photos Libraries, use:
# FORCE_REBUILD_INVENTORY_KEYS = {"backup_20250317", "current_default"}
FORCE_REBUILD_INVENTORY_KEYS = set()

# Set to True only when you want to pick Photos Library paths again.
# If False, Test 2 reuses paths saved in data/local_config/test2_library_paths.json.
FORCE_RESELECT_LIBRARY_PATHS = False

TEST2_LIBRARY_PROMPTS = {
    "backup_20250317": "Select BACKUP Photos Library: backup_20250317",
    "current_default": "Select CURRENT default Photos Library: current_default",
}

TEST2_DEFAULT_INITIAL_DIRS = {
    "backup_20250317": "/Volumes",
    "current_default": str(Path.home() / "Pictures"),
}

In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

TEST2_LIBRARY_HISTORY_PATH = Path("data/local_config/test2_library_paths.json")


def get_test2_library_path(library_key):
    library_history = load_json_file(TEST2_LIBRARY_HISTORY_PATH, default={}) or {}
    saved_library_path = library_history.get(library_key)

    if (
        saved_library_path
        and Path(saved_library_path).exists()
        and not FORCE_RESELECT_LIBRARY_PATHS
    ):
        library_path = Path(saved_library_path)

        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(f"{library_key} library path:", library_path)
        print()

        return library_path

    if saved_library_path:
        initial_dir = Path(saved_library_path).parent
    else:
        initial_dir = Path(TEST2_DEFAULT_INITIAL_DIRS.get(library_key, "/Volumes"))

    prompt = TEST2_LIBRARY_PROMPTS.get(
        library_key,
        f"Select Photos Library for: {library_key}",
    )

    print("=" * 80)
    print(prompt)
    print("=" * 80)

    library_path = Path(
        choose_photos_library_path(
            initial_dir=initial_dir,
            prompt=prompt,
        )
    )

    library_history[library_key] = str(library_path)
    library_history[f"{library_key}_selected_at"] = datetime.now().isoformat()
    save_json_file(TEST2_LIBRARY_HISTORY_PATH, library_history)

    print(f"{library_key} library path:", library_path)
    print()

    return library_path


def load_or_build_inventory(library_key):
    library_path = get_test2_library_path(library_key)
    should_rebuild_inventory = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild_inventory:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            return load_inventory_cache(library_key)
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    if should_rebuild_inventory:
        print("=" * 80)
        print(f"Force rebuild inventory: {library_key}")
        print("=" * 80)
    else:
        print("=" * 80)
        print(f"Build inventory: {library_key}")
        print("=" * 80)

    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Use saved Photos Library path for: backup_20250317
backup_20250317 library path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary

Load inventory cache: backup_20250317
loaded inventory cache: data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.56
inventory assets: 71573
inventory albums: 5170
inventory folders: 35
special assets:
  PATH_NONE: 0
  SYNDICATED: 0
  UNKNOWN_PATH: 0
movies: 6224
hidden: 0
favorites: 701
descriptions: 727
keywords: 23747

Use saved Photos Library path for: current_default
current_default library path: /Users/huohsien/Pictures/Photos Library.photoslibrary

Load inventory cache: current_default
loaded inventory cache: data/inventory_cache/current_default.inventory.pkl.gz
elapsed seconds: 0.75
inventory assets: 94701
inventory albums: 5942
inventory folders: 33
special 

In [3]:
# ============================================================
# Section 3: Photo Library asset unique ID audit
# ============================================================

print("=" * 120)
print("Section 3: Photo Library asset unique ID audit")
print("=" * 120)

print()
backup_unique_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    label="BACKUP",
)

print()
current_unique_ok = audit_photo_library_asset_unique_ids(
    inventory_current,
    label="CURRENT",
)

if not backup_unique_ok:
    raise RuntimeError("BACKUP photo_library_asset_unique_id audit failed.")

if not current_unique_ok:
    raise RuntimeError("CURRENT photo_library_asset_unique_id audit failed.")

print()
print("Photo Library asset unique ID audit passed.")

Section 3: Photo Library asset unique ID audit

BACKUP
--------------------------------------------------------------------------------
total asset count: 71573
generated unique ID count: 71573
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

CURRENT
--------------------------------------------------------------------------------
total asset count: 94701
generated unique ID count: 94701
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

Photo Library asset unique ID audit passed.


In [ ]:
# ============================================================
# Section 4: Run inventory comparison summary
# ============================================================

if not (backup_unique_ok and current_unique_ok):
    raise RuntimeError(
        "Photo Library asset unique ID audit failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
)

summarize_diff_records(diff_records)

diff record count: 60526

change_type counts
--------------------------------------------------------------------------------
ALBUM_FOLDER_PATHS_CHANGED: 317
ALBUM_MISSING_FROM_CURRENT: 68
ALBUM_NEW_IN_CURRENT: 841
ASSET_ALBUM_MEMBERSHIP_CHANGED: 12622
ASSET_FOLDER_PATHS_CHANGED: 19169
ASSET_METADATA_CHANGED: 3959
ASSET_MISSING_FROM_CURRENT: 14
ASSET_NEW_IN_CURRENT: 23335
ASSET_PRESENT_IN_CURRENT_AS_SYNDICATED: 193
FOLDER_MISSING_FROM_CURRENT: 5
FOLDER_NEW_IN_CURRENT: 3

scope counts
--------------------------------------------------------------------------------
album: 1226
asset: 59292
folder: 8


In [ ]:
# ============================================================
# Section 5: Print ASSET_MISSING_FROM_CURRENT review list
# ============================================================

missing_current_review_result = print_missing_current_review(
    diff_records=diff_records,
    inventory_current=inventory_current,
    max_current_candidates=5,
)

# Optional: write text/TSV files only when you explicitly want files.
WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES = False

if WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES:
    missing_current_review_file_result = write_missing_current_review_report(
        diff_records=diff_records,
        inventory_current=inventory_current,
        output_dir=Path("reports/test2_missing_current_review"),
        report_name_prefix="asset_missing_from_current_review",
    )
    print_missing_current_review_report_summary(missing_current_review_file_result)


Section 5A: Compact ASSET_MISSING_FROM_CURRENT review list
matched record count: 14

Quick counts
------------------------------------------------------------------------------------------------------------------------
is_movie: {False: 14}
hasadjustments: {False: 11, True: 3}
favorite: {False: 14}
hidden: {False: 14}
path_exists: {True: 14}

One-by-one manual verification checklist

01. IMG_0106.JPG
------------------------------------------------------------------------------------------------------------------------
change_type: ASSET_MISSING_FROM_CURRENT
description: Asset exists in backup normal assets, and no matching current normal asset or current syndicated asset was found.
note: Asset exists in backup normal assets, and no matching current normal asset or current syndicated asset was found.
uuid: 182137B0-D3F4-46FC-9499-0BB32E977ED9
unique_id: ('IMG_0106.JPG', '03-05 08:46:37.73', 131592, None, None)
date: 2025-03-05T08:46:37.732606+08:00
date_added: 2025-03-05T08:46:37.73526

In [ ]:
# # ============================================================
# # Appendix A: Duplicate diagnostic archive
# # 
# # TEMP: Diagnose potential duplicate groups by SHA256
# #       with full manual-review metadata
# # ============================================================

# import hashlib
# import os
# import time
# from datetime import datetime


# def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
#     cached_sha256 = asset.get("content_sha256")
#     if cached_sha256:
#         return cached_sha256

#     path = asset.get("path")

#     if path is None:
#         return None

#     if not os.path.exists(path):
#         return None

#     sha256 = hashlib.sha256()

#     with open(path, "rb") as f:
#         while True:
#             chunk = f.read(chunk_size)

#             if not chunk:
#                 break

#             sha256.update(chunk)

#     digest = sha256.hexdigest()
#     asset["content_sha256"] = digest
#     return digest


# def build_potential_duplicate_groups_by_unique_id(inventory):
#     unique_id_to_assets = {}

#     for asset in inventory["assets"]:
#         unique_id = asset.get("photo_library_asset_unique_id")

#         if unique_id is None:
#             continue

#         if unique_id not in unique_id_to_assets:
#             unique_id_to_assets[unique_id] = []

#         unique_id_to_assets[unique_id].append(asset)

#     return {
#         unique_id: assets
#         for unique_id, assets in unique_id_to_assets.items()
#         if len(assets) > 1
#     }


# def normalize_string_list(values):
#     result = []

#     if values is None:
#         return result

#     if isinstance(values, str):
#         return [values]

#     if isinstance(values, dict):
#         iterable = values.values()
#     elif isinstance(values, (list, tuple, set)):
#         iterable = values
#     else:
#         return [str(values)]

#     for item in iterable:
#         if item is None:
#             continue

#         if isinstance(item, str):
#             value = item
#         elif isinstance(item, dict):
#             value = (
#                 item.get("title")
#                 or item.get("name")
#                 or item.get("path")
#                 or item.get("folder_path")
#                 or item.get("album_path")
#             )
#         else:
#             value = str(item)

#         if value:
#             result.append(value)

#     return sorted(set(result))


# def get_asset_album_titles(asset):
#     albums = asset.get("albums")
#     return normalize_string_list(albums)


# def get_asset_folder_paths(asset):
#     folders = asset.get("folders")
#     folder_paths = normalize_string_list(folders)

#     # Some inventory formats may store folder paths under different keys.
#     extra_candidates = [
#         asset.get("folder_paths"),
#         asset.get("folder_path"),
#         asset.get("album_folder_paths"),
#     ]

#     for candidate in extra_candidates:
#         folder_paths.extend(normalize_string_list(candidate))

#     return sorted(set(folder_paths))


# def get_asset_keywords(asset):
#     keywords = asset.get("keywords")
#     return normalize_string_list(keywords)


# def get_asset_description(asset):
#     return (
#         asset.get("description")
#         or asset.get("caption")
#         or asset.get("title")
#         or ""
#     )


# def parse_date_added_for_sort(asset):
#     date_added = asset.get("date_added")

#     if not date_added:
#         return datetime.max

#     if isinstance(date_added, datetime):
#         return date_added

#     text = str(date_added)

#     try:
#         return datetime.fromisoformat(text.replace("Z", "+00:00"))
#     except Exception:
#         return datetime.max


# def asset_metadata_signature(asset):
#     return {
#         "albums": tuple(get_asset_album_titles(asset)),
#         "folders": tuple(get_asset_folder_paths(asset)),
#         "keywords": tuple(get_asset_keywords(asset)),
#         "description": get_asset_description(asset),
#         "favorite": asset.get("favorite"),
#         "hidden": asset.get("hidden"),
#         "hasadjustments": asset.get("hasadjustments"),
#         "adjustment_signature": asset.get("adjustment_signature"),
#     }


# def metadata_score(asset):
#     return (
#         len(get_asset_album_titles(asset)) * 10
#         + len(get_asset_folder_paths(asset)) * 10
#         + len(get_asset_keywords(asset)) * 5
#         + (1 if get_asset_description(asset) else 0)
#         + (1 if asset.get("favorite") else 0)
#         + (1 if asset.get("hidden") else 0)
#     )


# def choose_representative_asset(assets):
#     # Prefer metadata-rich assets; tie-break by earliest Date Added.
#     return sorted(
#         assets,
#         key=lambda asset: (
#             -metadata_score(asset),
#             parse_date_added_for_sort(asset),
#             asset.get("uuid") or "",
#         ),
#     )[0]


# def print_asset_manual_review_block(asset, indent="  "):
#     print(f"{indent}UUID:", asset.get("uuid"))
#     print(f"{indent}Original File Name:", asset.get("original_filename"))
#     print(f"{indent}Filename:", asset.get("filename"))
#     print(f"{indent}Date:", asset.get("date"))
#     print(f"{indent}Date Added:", asset.get("date_added"))
#     print(f"{indent}File Size:", asset.get("file_size_bytes"))
#     print(f"{indent}Has Adjustments:", asset.get("hasadjustments"))
#     print(f"{indent}Adjustment Signature:", asset.get("adjustment_signature"))
#     print(f"{indent}Width x Height:", asset.get("width"), "x", asset.get("height"))
#     print(f"{indent}Original Width x Height:", asset.get("original_width"), "x", asset.get("original_height"))
#     print(f"{indent}Albums:", get_asset_album_titles(asset))
#     print(f"{indent}Folder Paths:", get_asset_folder_paths(asset))
#     print(f"{indent}Keywords:", get_asset_keywords(asset))
#     print(f"{indent}Description:", get_asset_description(asset))
#     print(f"{indent}Favorite:", asset.get("favorite"))
#     print(f"{indent}Hidden:", asset.get("hidden"))
#     print(f"{indent}Path:", asset.get("path"))


# def print_cleanup_recommendation(assets):
#     metadata_signatures = [asset_metadata_signature(asset) for asset in assets]
#     metadata_all_same = all(
#         signature == metadata_signatures[0]
#         for signature in metadata_signatures
#     )

#     representative = choose_representative_asset(assets)

#     if metadata_all_same:
#         print("Recommendation:")
#         print("  Metadata appears identical.")
#         print("  Keep earliest / representative asset:")
#         print("   ", representative.get("uuid"))
#         print("  Delete other duplicate asset(s):")
#         for asset in assets:
#             if asset is not representative:
#                 print("   ", asset.get("uuid"))
#     else:
#         print("Recommendation:")
#         print("  Metadata differs across duplicate assets.")
#         print("  Do NOT blindly delete.")
#         print("  Suggested representative, based on richer metadata + earliest Date Added:")
#         print("   ", representative.get("uuid"))
#         print("  Before deleting others, manually confirm whether album/folder/keyword membership should be preserved.")


# def diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory,
#     label,
#     max_true_duplicate_groups_to_print=50,
#     max_key_collision_groups_to_print=20,
# ):
#     start_time = time.perf_counter()

#     potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

#     true_content_duplicate_groups = []
#     key_collision_groups = []
#     sha_error_assets = []

#     checked_asset_count = 0

#     for unique_id, assets in potential_groups.items():
#         sha256_to_assets = {}

#         for asset in assets:
#             checked_asset_count += 1
#             sha256 = compute_sha256_for_asset(asset)

#             if sha256 is None:
#                 sha_error_assets.append(asset)
#                 continue

#             if sha256 not in sha256_to_assets:
#                 sha256_to_assets[sha256] = []

#             sha256_to_assets[sha256].append(asset)

#         duplicate_sha_groups = {
#             sha256: sha_assets
#             for sha256, sha_assets in sha256_to_assets.items()
#             if len(sha_assets) > 1
#         }

#         if duplicate_sha_groups:
#             for sha256, sha_assets in duplicate_sha_groups.items():
#                 true_content_duplicate_groups.append(
#                     {
#                         "unique_id": unique_id,
#                         "sha256": sha256,
#                         "assets": sha_assets,
#                     }
#                 )

#         if len(sha256_to_assets) > 1:
#             key_collision_groups.append(
#                 {
#                     "unique_id": unique_id,
#                     "sha256_to_assets": sha256_to_assets,
#                 }
#             )

#     elapsed = time.perf_counter() - start_time

#     print(label)
#     print("-" * 120)
#     print("potential duplicate unique_id group count:", len(potential_groups))
#     print("checked asset count:", checked_asset_count)
#     print("sha error asset count:", len(sha_error_assets))
#     print("true content duplicate group count:", len(true_content_duplicate_groups))
#     print("key collision group count:", len(key_collision_groups))
#     print("elapsed seconds:", round(elapsed, 3))

#     print()
#     print("TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW")
#     print("-" * 120)

#     for index, group in enumerate(true_content_duplicate_groups, start=1):
#         if index > max_true_duplicate_groups_to_print:
#             print("... more true content duplicate groups not printed")
#             break

#         assets_sorted = sorted(
#             group["assets"],
#             key=lambda asset: (
#                 parse_date_added_for_sort(asset),
#                 asset.get("uuid") or "",
#             ),
#         )

#         print("=" * 120)
#         print(f"Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256:", group["sha256"])
#         print("asset count:", len(assets_sorted))

#         first_asset = assets_sorted[0]
#         print("Original File Name:", first_asset.get("original_filename"))
#         print("Date:", first_asset.get("date"))
#         print("File Size:", first_asset.get("file_size_bytes"))
#         print("Adjustment Signature:", first_asset.get("adjustment_signature"))

#         union_albums = sorted(
#             set(
#                 album
#                 for asset in assets_sorted
#                 for album in get_asset_album_titles(asset)
#             )
#         )
#         union_folders = sorted(
#             set(
#                 folder
#                 for asset in assets_sorted
#                 for folder in get_asset_folder_paths(asset)
#             )
#         )
#         union_keywords = sorted(
#             set(
#                 keyword
#                 for asset in assets_sorted
#                 for keyword in get_asset_keywords(asset)
#             )
#         )

#         print("Union Albums:", union_albums)
#         print("Union Folder Paths:", union_folders)
#         print("Union Keywords:", union_keywords)

#         print()
#         print_cleanup_recommendation(assets_sorted)
#         print()

#         for asset_index, asset in enumerate(assets_sorted, start=1):
#             print("-" * 120)
#             print(f"Asset {asset_index}")
#             print_asset_manual_review_block(asset, indent="  ")

#         print()

#     print()
#     print("KEY COLLISION GROUPS")
#     print("-" * 120)

#     for index, group in enumerate(key_collision_groups, start=1):
#         if index > max_key_collision_groups_to_print:
#             print("... more key collision groups not printed")
#             break

#         print("=" * 120)
#         print(f"Key Collision Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256 count:", len(group["sha256_to_assets"]))

#         for sha256, assets in group["sha256_to_assets"].items():
#             print("  sha256:", sha256)
#             print("  asset count:", len(assets))

#             for asset in assets:
#                 print("    uuid:", asset.get("uuid"))
#                 print("    original_filename:", asset.get("original_filename"))
#                 print("    filename:", asset.get("filename"))
#                 print("    date:", asset.get("date"))
#                 print("    date_added:", asset.get("date_added"))
#                 print("    file_size_bytes:", asset.get("file_size_bytes"))
#                 print("    albums:", get_asset_album_titles(asset))
#                 print("    folder_paths:", get_asset_folder_paths(asset))
#                 print("    keywords:", get_asset_keywords(asset))
#                 print("    path:", asset.get("path"))

#         print()

#     return {
#         "potential_groups": potential_groups,
#         "true_content_duplicate_groups": true_content_duplicate_groups,
#         "key_collision_groups": key_collision_groups,
#         "sha_error_assets": sha_error_assets,
#     }


# backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_backup,
#     "BACKUP potential duplicate diagnostic with metadata",
# )

# print()

# current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_current,
#     "CURRENT potential duplicate diagnostic with metadata",
# )

BACKUP potential duplicate diagnostic with metadata
------------------------------------------------------------------------------------------------------------------------
potential duplicate unique_id group count: 0
checked asset count: 0
sha error asset count: 0
true content duplicate group count: 0
key collision group count: 0
elapsed seconds: 0.393

TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW
------------------------------------------------------------------------------------------------------------------------

KEY COLLISION GROUPS
------------------------------------------------------------------------------------------------------------------------

CURRENT potential duplicate diagnostic with metadata
------------------------------------------------------------------------------------------------------------------------
potential duplicate unique_id group count: 0
checked asset count: 0
sha error asset count: 0
true content duplicate group count: 0
key collision group count: 

In [ ]:
# # ============================================================
# # Appendix B: Debug assets without unique ID
# # 
# # DEBUG: Dump assets without photo_library_asset_unique_id
# # ============================================================

# import os
# import time
# from collections import Counter

# def debug_dump_assets_without_photo_library_asset_unique_id(inventory, label, max_print=80):
#     missing_assets = [
#         asset
#         for asset in inventory["assets"]
#         if asset.get("photo_library_asset_unique_id") is None
#     ]

#     reason_counter = Counter()

#     print(label)
#     print("-" * 120)
#     print("assets without photo_library_asset_unique_id:", len(missing_assets))
#     print()

#     for asset in missing_assets:
#         path = asset.get("path")
#         original_filename = asset.get("original_filename")
#         filename = asset.get("filename")
#         date = asset.get("date")
#         file_size_bytes = asset.get("file_size_bytes")
#         adjustment_signature = asset.get("adjustment_signature")

#         if original_filename is None and filename is None:
#             reason_counter["missing filename and original_filename"] += 1

#         if date is None:
#             reason_counter["missing date"] += 1

#         if path is None:
#             reason_counter["path is None"] += 1
#         elif not os.path.exists(path):
#             reason_counter["path does not exist"] += 1

#         if file_size_bytes is None:
#             reason_counter["file_size_bytes is None"] += 1

#         if adjustment_signature is None:
#             reason_counter["adjustment_signature is None"] += 1

#     print("reason counter:")
#     for reason, count in reason_counter.most_common():
#         print(f"  {reason}: {count}")

#     print()
#     print("missing asset details:")
#     print("-" * 120)

#     for index, asset in enumerate(missing_assets[:max_print], start=1):
#         path = asset.get("path")

#         print(f"{index:02d}.")
#         print("  uuid:", asset.get("uuid"))
#         print("  original_filename:", asset.get("original_filename"))
#         print("  filename:", asset.get("filename"))
#         print("  date:", asset.get("date"))
#         print("  date_added:", asset.get("date_added"))
#         print("  path:", path)
#         print("  path_exists:", None if path is None else os.path.exists(path))
#         print("  file_size_bytes:", asset.get("file_size_bytes"))
#         print("  adjustment_signature:", asset.get("adjustment_signature"))
#         print("  is_movie:", asset.get("is_movie"))
#         print("  hasadjustments:", asset.get("hasadjustments"))
#         print("  path_edited:", asset.get("path_edited"))
#         print("  asset_scope:", asset.get("asset_scope"))
#         print("  albums:", list((asset.get("albums") or {}).values()))
#         print("  folders:", list((asset.get("folders") or {}).values()))
#         print("-" * 120)

# debug_dump_assets_without_photo_library_asset_unique_id(
#     inventory_current,
#     "CURRENT DEFAULT assets without photo_library_asset_unique_id",
# )

CURRENT DEFAULT assets without photo_library_asset_unique_id
------------------------------------------------------------------------------------------------------------------------
assets without photo_library_asset_unique_id: 0

reason counter:

missing asset details:
------------------------------------------------------------------------------------------------------------------------
